In [1]:
import os, time, warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
start = time.time()

In [3]:
df = pd.read_csv('flights.csv')
print("Raw shape:", df.shape)

Raw shape: (336776, 21)


In [4]:
df = df[df['arr_delay'].notna()]
for col in ['arr_delay','dep_delay']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=['arr_delay'])

In [5]:
df = df[(df['arr_delay'] > -300) & (df['arr_delay'] < 500)]
if 'dep_delay' in df.columns:
    df = df[(df['dep_delay'] > -300) & (df['dep_delay'] < 500) | df['dep_delay'].isna()]


In [6]:
MAX_ROWS = 120000
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)
    print("Downsampled to", df.shape)

Downsampled to (120000, 21)


In [7]:
if 'time_hour' in df.columns:
    df['time_hour'] = pd.to_datetime(df['time_hour'], errors='coerce')
    df['hour'] = df['time_hour'].dt.hour.fillna(-1).astype(int)
    df['dayofweek'] = df['time_hour'].dt.dayofweek.fillna(-1).astype(int)
else:
    if 'sched_dep_time' in df.columns:
        def parse_hhmm(x):
            try:
                x = int(x)
                return x // 100 if 0 <= (x // 100) < 24 else -1
            except:
                return -1
        df['hour'] = df['sched_dep_time'].apply(parse_hhmm)
        df['dayofweek'] = -1
    else:
        df['hour'] = -1
        df['dayofweek'] = -1

df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)

In [8]:
if 'origin' in df.columns and 'dest' in df.columns:
    df['route'] = df['origin'].astype(str) + "_" + df['dest'].astype(str)
    route_stats = df.groupby('route')['arr_delay'].agg(['mean','median','count']).rename(
        columns={'mean':'route_arr_mean','median':'route_arr_median','count':'route_count'})
    df = df.merge(route_stats, how='left', on='route')
else:
    df['route_arr_mean'] = df['arr_delay'].mean()
    df['route_arr_median'] = df['arr_delay'].median()
    df['route_count'] = 0

In [9]:
for c in ['origin','dest','carrier']:
    if c in df.columns:
        freq = df[c].value_counts().to_dict()
        df[f'{c}_freq'] = df[c].map(freq).fillna(0)
    else:
        df[f'{c}_freq'] = 0

In [10]:
if 'dep_delay' in df.columns:
    df['dep_delay_missing'] = df['dep_delay'].isna().astype(int)
    df['dep_delay'] = df['dep_delay'].fillna(df['dep_delay'].median())# median is robust to outlier
else:
    df['dep_delay'] = 0
    df['dep_delay_missing'] = 1


In [11]:
for c in ['time_hour','route','sched_dep_time']:
    if c in df.columns:
        df.drop(columns=[c], inplace=True)

In [12]:
features = [
    'dep_delay', 'hour', 'dayofweek', 'is_weekend',
    'route_arr_mean', 'route_arr_median', 'route_count',
    'origin_freq', 'dest_freq', 'carrier_freq',
    'dep_delay_missing'
]
features = [f for f in features if f in df.columns]
X = df[features].copy()
y = df['arr_delay'].copy()

In [14]:
from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy='median')
X = pd.DataFrame(imp.fit_transform(X), columns=X.columns)

In [15]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train/test sizes:", X_train.shape, X_test.shape)

Train/test sizes: (96000, 11) (24000, 11)


In [17]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

rf = RandomForestRegressor(random_state=42, n_jobs=-1)
hgb = HistGradientBoostingRegressor(random_state=42)
rf_params = {'n_estimators':[100,150], 'max_depth':[6,10,None], 'min_samples_leaf':[1,3]}
hgb_params = {'max_iter':[100,200], 'max_depth':[3,6,None], 'learning_rate':[0.05,0.1]}

In [18]:
from sklearn.model_selection import RandomizedSearchCV

rf_search = RandomizedSearchCV(rf, rf_params, n_iter=6, scoring='r2', cv=3, random_state=42, n_jobs=-1)
hgb_search = RandomizedSearchCV(hgb, hgb_params, n_iter=6, scoring='r2', cv=3, random_state=42, n_jobs=-1)

In [19]:
print(" RF ...")
rf_search.fit(X_train, y_train)
print("HGB ...")
hgb_search.fit(X_train, y_train)

 RF ...
HGB ...


RandomizedSearchCV(cv=3,
                   estimator=HistGradientBoostingRegressor(random_state=42),
                   n_iter=6, n_jobs=-1,
                   param_distributions={'learning_rate': [0.05, 0.1],
                                        'max_depth': [3, 6, None],
                                        'max_iter': [100, 200]},
                   random_state=42, scoring='r2')

In [21]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
for name, estimator in [('RandomForest', rf_search.best_estimator_), ('HGB', hgb_search.best_estimator_)]:
    pred = estimator.predict(X_test)
    print(f"{name} -> MSE: {mean_squared_error(y_test, pred):.3f}, R2: {r2_score(y_test, pred):.3f}")

RandomForest -> MSE: 305.961, R2: 0.841
HGB -> MSE: 301.548, R2: 0.843


In [23]:
import joblib

best = rf_search.best_estimator_ if r2_score(y_test, rf_search.predict(X_test)) >= r2_score(y_test, hgb_search.predict(X_test)) else hgb_search.best_estimator_
joblib.dump(best, "best_flight_delay_model.joblib")
print("Saved model to best_flight_delay_model.joblib")
pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1).head(500).to_csv("engineered_features_sample.csv", index=False)
print("Saved sample engineered features to engineered_features_sample.csv")
print("Elapsed:", time.time()-start)

Saved model to best_flight_delay_model.joblib
Saved sample engineered features to engineered_features_sample.csv
Elapsed: 673.3529903888702
